# Plot gapfilled time series for three stations

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, correlate, correlation_lags, find_peaks_cwt
from scipy.integrate import trapz
from datetime import datetime, timedelta
from vtools.functions.filter import cosine_lanczos
from vtools.data.vtime import days, hours, minutes
import cmocean

/global/home/users/jennaisrael/.conda/envs/schimpy/lib/python3.7/site-packages/dask/dataframe/utils.py:15: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


In [10]:
#function comes from schimpy metricsplot.py script https://github.com/CADWRDeltaModeling/schimpy/blob/master/schimpy/metricsplot.py
def filter_timeseries(tss, cutoff_period=hours(40)):
    """ Filter time series

        Parameters
        ----------

        Returns
        -------
        list of vtools.data.timeseries.TimeSeries
            filtered time series
    """

    filtered = []
    ts=tss #try removing the loop for now
    if ts is None:
        filtered.append(None)
    else:
        #print(ts)
        ts_filtered = cosine_lanczos(ts, cutoff_period=cutoff_period)
        ts_filtered.filtered = 'cosine_lanczos'
        #ts_filtered.unit = ts.unit
        #filtered.append(ts_filtered)
    # for ts in tss:
    #     if ts is None:
    #         filtered.append(None)
    #     else:
    #         print(ts)
    #         ts_filtered = cosine_lanczos(ts, cutoff_period=cutoff_period)
    #         ts_filtered.filtered = 'cosine_lanczos'
    #         ts_filtered.unit = ts.unit
    #         filtered.append(ts_filtered)
    return ts_filtered

In [2]:
#load the SCHA filtered signal for Point Reyes 
#use Eli's SCHA filter subtide that I then subtracted the 40 day box car from
ptreyes=pd.read_csv("/global/scratch/users/jennaisrael/time_varying_data/tide_gauge_data/bcfiltered_SCHA_subtide_ptreyes.csv")
dtformat = '%Y-%m-%dT%H:%M:%S'
ptreyes['datetime'] = pd.to_datetime(ptreyes['datetime'],format=dtformat)
pr_filt=ptreyes[['datetime','box_40d_filt']].rename(columns={"box_40d_filt": "Residual"}).dropna()
pr_filt.set_index("datetime",inplace=True)
pr_filt

,Residual
datetime,
2012-02-12 01:30:00,0.057819
2012-02-12 02:00:00,0.059018
2012-02-12 02:30:00,0.060316
2012-02-12 03:00:00,0.061613
2012-02-12 03:30:00,0.062910
...,...
2023-11-19 20:30:00,-0.110012
2023-11-19 21:00:00,-0.109173
2023-11-19 21:30:00,-0.108133


In [11]:
jp=pd.read_csv('/global/scratch/users/jennaisrael/time_varying_data/dmsdatastore/salinity/jp_gapfilled_2025_12_15.csv')
jp['datetime']=pd.to_datetime(jp['datetime'],format=dtformat)
jp.set_index("datetime",inplace=True)
jp=jp.asfreq('15min')
jp_filt=filter_timeseries(jp)

frk=pd.read_csv('/global/scratch/users/jennaisrael/time_varying_data/dmsdatastore/salinity/frk_gapfilled_2026_01_26.csv')
frk['datetime']=pd.to_datetime(frk['datetime'],format=dtformat)
frk.set_index("datetime",inplace=True)
frk=frk.asfreq('15min')
frk_filt=filter_timeseries(frk)

hol2=pd.read_csv('/global/scratch/users/jennaisrael/time_varying_data/dmsdatastore/salinity/hol2_gapfilled_2025_12_15.csv')
hol2['datetime']=pd.to_datetime(hol2['datetime'],format=dtformat)
hol2.set_index("datetime",inplace=True)
hol2=hol2.asfreq('15min')
hol2_filt=filter_timeseries(hol2)

hol2

,Salinity[microS/cm]
datetime,
2009-09-03 12:00:00,722.0
2009-09-03 12:15:00,723.0
2009-09-03 12:30:00,725.0
2009-09-03 12:45:00,726.0
2009-09-03 13:00:00,725.0
...,...
2025-09-07 22:00:00,591.0
2025-09-07 22:15:00,581.0
2025-09-07 22:30:00,596.0


In [ ]:
fig, ax = plt.subplots(layout="constrained",figsize=(11,6))#,gridspec_kw={'height_ratios': [3, 1, 1]})

#Set the date limits of the plot HERE so I can rescale the axes by min and max in this range 
lim1=datetime(2017,7,1)
lim2=datetime(2017,11,1)
#colors to be consistent with other plors
colors = plt.colormaps['Dark2'].colors #this is a tuple so need to use tuple indexing [rom][column]


a, = ax.plot(jp_filt,color=cmocean.cm.ice(3/6),label="Jersey Point")

b, = ax.plot(frk_filt,color=cmocean.cm.ice(4/6),label="Franks Tract")
#ax[0].scatter(df_f_wo21.datetime.loc[peak_frk_both],df_f_wo21.frk.loc[peak_frk_both],marker="x",color='darkgreen')
c, = ax.plot(hol2_filt,color=cmocean.cm.ice(5/6),label="Holland Cut")
ax.set_ylabel(r'Salinity [$\mu S/cm$]')
ax.set_ylim(0,450)
# ax0=ax.twinx()
# d, = ax0.plot(pr_filt,color='grey', label="Point Reyes")
# ax0.set_ylabel("Residual [m]",color=cmocean.cm.ice(0))
# ax.set_title('Salinity and Non Tidal Residual Anomaly')
ax.set_xlim(lim1,lim2)
# ax0.set_ylim(-0.2,0.2)
fig.legend()
#leg = plt.legend(handles=[a,b,c,d,e,f,g],loc='upper right', bbox_to_anchor=(1.8, 2.0))
plt.show()


In [ ]:
# what is the mean flow of the sac vs sj in this period

In [2]:
#read in the flux data so I can plot outflow
flux1=pd.read_csv('/global/scratch/users/jennaisrael/climate_data_processing/sl_regression/flux_from_bdschism_2025_06_05.csv',parse_dates=["datetime"],dtype=np.float32).set_index(["datetime"])#, names=['time','coyote','ccc_rock','ccc_old','swp','cvp','sjr','calaveras','east','american','sac','yolo_toedrain','yolo','northbay','napa','ccc_victoria'])
flux1


,coyote,ccc_rock,ccc_old,swp,cvp,sjr,calaveras,east,american,sac,yolo_toedrain,yolo,northbay,napa,ccc_victoria
datetime,,,,,,,,,,,,,,,
2006-10-01 00:00:00,-0.62,0.67,4.52,262.320007,120.830002,-94.860001,0.0,-25.780001,-72.769997,-330.570007,0.68,0.00,2.48,-0.01,0.0
2006-10-01 00:15:00,-0.62,0.67,4.52,263.239990,120.820000,-94.860001,0.0,-25.730000,-71.919998,-330.640015,-4.62,0.00,2.46,-0.01,0.0
2006-10-01 00:30:00,-0.62,0.66,4.52,263.410004,120.820000,-95.430000,0.0,-25.690001,-73.339996,-331.000000,-8.10,0.00,2.46,-0.01,0.0
2006-10-01 00:45:00,-0.62,0.66,4.52,263.390015,120.820000,-94.860001,0.0,-25.660000,-73.910004,-331.070007,-9.66,0.00,2.45,-0.01,0.0
2006-10-01 01:00:00,-0.62,0.66,4.52,263.350006,120.820000,-94.860001,0.0,-25.629999,-73.910004,-331.420013,-10.65,0.00,2.45,-0.01,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-05-13 23:00:00,-0.84,2.46,4.18,1.680000,24.850000,-179.250000,0.0,-63.320000,-116.379997,-749.340027,0.06,-0.01,1.55,-1.68,0.0
2024-05-13 23:15:00,-0.84,2.44,4.22,1.700000,24.719999,-180.100006,0.0,-63.450001,-115.820000,-749.340027,0.06,-0.01,1.55,-1.65,0.0
2024-05-13 23:30:00,-0.84,2.41,4.27,1.720000,24.570000,-179.529999,0.0,-63.360001,-116.379997,-749.340027,0.06,-0.01,1.55,-1.65,0.0


In [4]:
flux_oct2018=flux1.loc[pd.to_datetime('2018-09-01'):pd.to_datetime('2018-11-01')]
flux_oct2018

,coyote,ccc_rock,ccc_old,swp,cvp,sjr,calaveras,east,american,sac,yolo_toedrain,yolo,northbay,napa,ccc_victoria
datetime,,,,,,,,,,,,,,,
2018-09-01 00:00:00,-0.39,0.0,1.5,163.600006,120.349998,-19.540001,0.0,-1.49,-53.240002,-462.779999,-9.57,-0.01,2.80,0.0,3.81
2018-09-01 00:15:00,-0.39,0.0,1.5,176.000000,120.349998,-19.540001,0.0,-1.49,-53.240002,-462.750000,-10.68,-0.01,2.76,0.0,3.83
2018-09-01 00:30:00,-0.40,0.0,1.5,186.960007,120.360001,-19.540001,0.0,-1.48,-52.950001,-462.720001,-10.00,-0.01,2.74,0.0,3.84
2018-09-01 00:45:00,-0.40,0.0,1.5,194.490005,120.360001,-19.540001,0.0,-1.48,-52.950001,-462.690002,-11.24,-0.01,2.73,0.0,3.85
2018-09-01 01:00:00,-0.40,0.0,1.5,198.520004,120.360001,-19.540001,0.0,-1.47,-53.240002,-462.660004,-10.59,-0.01,2.72,0.0,3.86
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-10-31 23:00:00,-0.35,0.0,0.0,31.200001,97.669998,-60.880001,0.0,-12.92,-45.590000,-207.360001,7.70,0.00,2.52,0.0,4.88
2018-10-31 23:15:00,-0.36,0.0,0.0,34.009998,97.669998,-60.880001,0.0,-13.24,-45.590000,-206.850006,7.82,0.00,2.53,0.0,4.89
2018-10-31 23:30:00,-0.36,0.0,0.0,39.020000,97.660004,-60.880001,0.0,-13.59,-45.590000,-206.910004,7.87,0.00,2.53,0.0,4.90


In [5]:
print(flux_oct2018['sac'].mean())
print(flux_oct2018['sjr'].mean())

-346.77227783203125
-41.09272003173828


In [6]:
-346/-41

8.439024390243903